In [ ]:
#Pranav Anand
#23MIC0006

In [ ]:
import os
import numpy as np
import faiss

from pypdf import PdfReader
from docx import Document
from sentence_transformers import SentenceTransformer


# --------------------------------------------------
# 1. Extract text from PDF / DOCX / TXT
# --------------------------------------------------

def extract_text(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".pdf":
        reader = PdfReader(file_path)
        text = ""

        for page in reader.pages:
            text += page.extract_text() or ""

        return text

    elif extension == ".docx":
        document = Document(file_path)

        text = ""
        for paragraph in document.paragraphs:
            text += paragraph.text + "\n"

        return text

    elif extension == ".txt":
        with open(file_path, "r", encoding="utf-8") as file:
            return file.read()

    else:
        raise ValueError("Unsupported file type")


# --------------------------------------------------
# 2. Split text into chunks
# --------------------------------------------------

def split_text(text, chunk_size=500):
    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


# --------------------------------------------------
# 3. Load Sentence Transformer
# --------------------------------------------------

model = SentenceTransformer("all-MiniLM-L6-v2")


# --------------------------------------------------
# 4. Create embeddings and store in FAISS
# --------------------------------------------------

def create_index(chunks):
    embeddings = model.encode(
        chunks,
        convert_to_numpy=True
    )

    # Normalize embeddings
    faiss.normalize_L2(embeddings)

    # all-MiniLM-L6-v2 produces 384-dimensional embeddings
    dimension = 384

    index = faiss.IndexFlatIP(dimension)

    index.add(embeddings)

    return index


# --------------------------------------------------
# 5. Search for relevant chunks
# --------------------------------------------------

def search(query, index, chunks, top_k=1):

    # Convert question into an embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    # Normalize query embedding
    faiss.normalize_L2(query_embedding)

    # Search FAISS
    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for i in range(top_k):
        chunk_index = indices[0][i]

        results.append({
            "chunk": chunks[chunk_index],
            "score": float(scores[0][i])
        })

    return results


# --------------------------------------------------
# 6. Main program
# --------------------------------------------------

file_path = "23MIC0006.pdf"

# Extract text
text = extract_text(file_path)

# Split text into chunks
chunks = split_text(text)

# Create FAISS index
index = create_index(chunks)

print("\nDocument processed successfully!")
print("Number of chunks:", len(chunks))

# Ask questions
while True:

    query = input("\nAsk a question (type 'exit' to quit): ")

    if query.lower() == "exit":
        break

    results = search(
        query,
        index,
        chunks,
        top_k=1
    )

    print("\nMost relevant chunk:")
    print(results[0]["chunk"])

    print("\nSimilarity score:",
          results[0]["score"])